# Trading Agent Broad Strategy Discovery

Reproducible companion for the 2026-08-14 Reddit-sourced strategy restart. This notebook reads the committed audit artifacts; the expensive raw replays remain in `scripts/run_*validation.py`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
def load(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

quality = load('data/external_nq_quality/EXTERNAL_NQ_QUALITY.json')
broad = load('data/broad_strategy_discovery/BROAD_STRATEGY_DISCOVERY.json')
long_run = load('data/long_history_strategy_validation/LONG_HISTORY_STRATEGY_VALIDATION.json')
daily_ibs = load('data/daily_ibs_validation/LONG_HISTORY_STRATEGY_VALIDATION.json')
regime = load('data/long_history_regime_selector/LONG_HISTORY_REGIME_SELECTOR.json')

## Data-quality decision

The public long-history file is never a paper provider. One mis-scaled interval is quarantined; its remaining RTH daily returns are checked against Yahoo.

In [ ]:
pd.Series({
    'audit_status': quality['status'],
    'raw_rows': quality['shape']['rows'],
    'quarantined_rows': quality['quality']['quarantined_rows'],
    'clean_rows': quality['quality']['clean_rows'],
    'yahoo_return_correlation': quality['yahoo_daily_agreement']['close_return_correlation'],
    'paper_provider_allowed': quality['paper_provider_allowed'],
})

## Family-level holdout results

One parameter variant per family is selected before the untouched holdout. The daily IBS family is appended from its isolated test because it was implemented after the first long-history pass.

In [ ]:
finalists = list(long_run['finalists']) + list(daily_ibs['finalists'])
rows = []
for item in finalists:
    rows.append({
        'family': item['family'],
        'variant': item['variant'],
        'long_n': item['external_all']['n'],
        'long_wr': item['external_all']['wr'],
        'holdout_n': item['external_holdout']['n'],
        'holdout_wr': item['external_holdout']['wr'],
        'holdout_pf': item['external_holdout']['pf'],
        'holdout_expectancy_r': item['external_holdout']['expectancy_r'],
        'paid_n': item['paid_recent']['n'],
        'paid_wr': item['paid_recent']['wr'],
        'yahoo_n': item['independent_yahoo_current']['n'],
        'yahoo_wr': item['independent_yahoo_current']['wr'],
        'paper_eligible': item['paper_eligible'],
    })
family_results = pd.DataFrame(rows).sort_values('holdout_wr', ascending=False)
family_results

In [ ]:
holdout_comparison = family_results[['family', 'holdout_n', 'holdout_wr', 'holdout_pf', 'holdout_expectancy_r']].copy()
holdout_comparison['target_wr'] = 0.70
holdout_comparison['gap_to_target'] = holdout_comparison['holdout_wr'] - holdout_comparison['target_wr']
holdout_comparison.sort_values('holdout_wr', ascending=False)

## Regime-router robustness result

The nonlinear selector was fit on development only, chose a fixed probability threshold on validation, and then opened the holdout. It cannot alter stops, targets, risk, or quantity.

In [ ]:
pd.DataFrame([
    {'period': period, **regime['external'][period]}
    for period in ('development', 'validation', 'holdout')
])[['period', 'n', 'wr', 'pf', 'expectancy_r', 'max_dd_r']]

## Promotion assertions

These assertions deliberately fail the research promotion decision closed if any report unexpectedly marks a candidate eligible without the reviewed gates being updated.

In [ ]:
assert broad['status'] == 'NO_70_PERCENT_STRATEGY_VALIDATED'
assert not any(row['paper_eligible'] for row in finalists)
assert regime['paper_eligible'] is False
assert quality['paper_provider_allowed'] is False
'PASS: research evidence remains fail-closed; no strategy was promoted.'